In [1]:
import math

import matplotlib.pyplot as plt
import torch


def periodic_hann(
    length: int,
    *,
    device=None,
    dtype=torch.float32,
) -> torch.Tensor:
    """
    Periodic Hann window:

        Hann_L[n] = 0.5 * (1 - cos(2*pi*n/L))

    for n = 0, ..., L - 1.
    """
    n = torch.arange(length, device=device, dtype=dtype)

    return 0.5 * (
        1.0 - torch.cos(2.0 * math.pi * n / length)
    )


def make_mauler_windows(
    analysis_length: int,
    synthesis_length: int,
    zero_length: int,
    hop_length: int,
    *,
    device=None,
    dtype=torch.float32,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Exact Mauler/Wang asymmetric analysis-synthesis window pair.

    Parameters
    ----------
    analysis_length:
        Full FFT/analysis length K.

    synthesis_length:
        Effective synthesis-window length 2M.

    zero_length:
        Initial zero section d in the analysis window.

    Returns
    -------
    analysis_window:
        Tensor of shape [K].

    synthesis_window:
        Tensor of shape [K], zero-padded on the left.
        Its effective support is the final 2M samples.

    Notes
    -----
    For perfect reconstruction, use hop_length = M,
    or an appropriate divisor of M with synthesis scaling.
    """

    K = analysis_length
    two_M = synthesis_length

    if two_M % 2 != 0:
        raise ValueError("synthesis_length must be even.")

    M = two_M // 2
    d = zero_length

    if d < 0:
        raise ValueError("zero_length must be non-negative.")

    if d > K - two_M:
        raise ValueError(
            "zero_length must satisfy "
            "zero_length <= analysis_length - synthesis_length."
        )

    # Length of the long Hann prototype.
    long_half_length = K - M - d
    long_hann_length = 2 * long_half_length

    hann_short = periodic_hann(
        two_M,
        device=device,
        dtype=dtype,
    )

    hann_long = periodic_hann(
        long_hann_length,
        device=device,
        dtype=dtype,
    )

    eps = torch.finfo(dtype).eps

    # -----------------------------------------------------
    # Analysis window h[n]
    #
    # Four sections:
    #   h1: d zeros
    #   h2: long rising sqrt-Hann
    #   h3: final M samples of that long rising sqrt-Hann
    #   h4: descending half of sqrt-Hann_{2M}
    # -----------------------------------------------------

    h1 = torch.zeros(
        d,
        device=device,
        dtype=dtype,
    )

    h2_length = K - two_M - d

    h2 = torch.sqrt(
        hann_long[:h2_length].clamp_min(0.0)
    )

    h3_indices = torch.arange(
        h2_length,
        h2_length + M,
        device=device,
    )

    h3 = torch.sqrt(
        hann_long[h3_indices].clamp_min(0.0)
    )

    h4 = torch.sqrt(
        hann_short[M:].clamp_min(0.0)
    )

    analysis_window = torch.cat(
        [h1, h2, h3, h4]
    )

    # -----------------------------------------------------
    # Synthesis window f[n]
    #
    # f1 and f2 are zero.
    #
    # f3 = Hann_{2M}[n] / h3[n]
    # f4 = sqrt(Hann_{2M}[n + M])
    # -----------------------------------------------------

    f1_f2 = torch.zeros(
        K - two_M,
        device=device,
        dtype=dtype,
    )

    f3 = hann_short[:M] / h3.clamp_min(eps)

    f4 = torch.sqrt(
        hann_short[M:].clamp_min(0.0)
    )

    synthesis_window = torch.cat(
        [f1_f2, f3, f4]
    )

    synthesis_window *= hop_length / M

    return analysis_window, synthesis_window

In [2]:
sample_rate = 44100

analysis_length = 512
synthesis_length = 128
hop_length = 64

analysis_window, synthesis_window = make_mauler_windows(
    analysis_length=analysis_length,
    synthesis_length=synthesis_length,
    zero_length = 0,
    hop_length=hop_length
)

In [ ]:
plt.figure(figsize=(11, 4))

plt.plot(
    analysis_window.cpu(),
    label="Asymmetric analysis window",
)

plt.plot(
    synthesis_window.cpu(),
    label="Short synthesis window, zero-padded",
)

plt.axvline(
    analysis_length - synthesis_length,
    linestyle="--",
    label="Start of synthesis support",
)

plt.xlabel("Sample inside FFT frame")
plt.ylabel("Amplitude")
plt.title("Exact Mauler analysis and synthesis windows")
plt.grid()
plt.legend()
plt.show()

In [ ]:
effective_window = analysis_window * synthesis_window

target_effective_window = torch.zeros_like(effective_window)
target_effective_window[-synthesis_length:] = periodic_hann(
    synthesis_length
)

max_error = (
    effective_window - target_effective_window
).abs().max()

print(f"Product error: {max_error.item():.3e}")

plt.figure(figsize=(11, 4))

plt.plot(
    effective_window.cpu(),
    label="Analysis × synthesis",
)

plt.plot(
    target_effective_window.cpu(),
    linestyle="--",
    label="Target periodic Hann",
)

plt.xlabel("Sample inside FFT frame")
plt.ylabel("Amplitude")
plt.title("Effective analysis-synthesis window")
plt.grid()
plt.legend()
plt.show()

# Transform

In [5]:
import torch
import torch.nn.functional as F


class CausalSTFT:
    def __init__(
        self,
        analysis_window: torch.Tensor,
        synthesis_window: torch.Tensor,
        hop_length: int,
    ):
        frame_length = analysis_window.numel()

        if hop_length > frame_length:
            raise ValueError(
                "hop_length cannot be larger than the frame length."
            )

        self.analysis_window = analysis_window
        self.synthesis_window = synthesis_window

        self.hop_length = hop_length
        self.frame_length = frame_length
        self.n_fft = frame_length
        self.left_padding = frame_length - hop_length

        self.streaming_input_buffer = None
        self.streaming_output_buffer = None

    def reset_streaming_stft(self):
        self.streaming_input_buffer = None

    def reset_streaming_istft(self):
        self.streaming_output_buffer = None

    def stft(
        self,
        audio: torch.Tensor,
        pad_end: bool = True,
    ) -> tuple[torch.Tensor, int]:
        original_length = audio.shape[-1]
        audio = audio.reshape(1, -1)

        window = self.analysis_window.to(
            device=audio.device,
            dtype=audio.dtype,
        )

        remainder = original_length % self.hop_length

        if pad_end:
            right_padding = (
                0
                if remainder == 0
                else self.hop_length - remainder
            )
        else:
            if original_length < self.hop_length:
                raise ValueError(
                    "The signal is shorter than one hop."
                )

            usable_length = (
                original_length // self.hop_length
            ) * self.hop_length

            audio = audio[..., :usable_length]
            right_padding = 0

        padded_audio = F.pad(
            audio,
            (self.left_padding, right_padding),
        )

        frames = padded_audio.unfold(
            dimension=-1,
            size=self.frame_length,
            step=self.hop_length,
        )

        frames = frames * window

        spectrum = torch.fft.rfft(
            frames,
            n=self.n_fft,
            dim=-1,
        )

        spectrum = spectrum.transpose(1, 2)

        return spectrum, original_length
    
    
    def streaming_stft(
        self,
        audio_hop: torch.Tensor,
    ) -> torch.Tensor:
        if audio_hop.ndim == 1:
            audio_hop = audio_hop.unsqueeze(0)

        if audio_hop.shape[-1] != self.hop_length:
            raise ValueError(
                f"Expected {self.hop_length} samples, "
                f"received {audio_hop.shape[-1]}."
            )

        batch_size = audio_hop.shape[0]

        if self.streaming_input_buffer is None:
            self.streaming_input_buffer = torch.zeros(
                batch_size,
                self.frame_length,
                device=audio_hop.device,
                dtype=audio_hop.dtype,
            )

        self.streaming_input_buffer[
            :,
            :-self.hop_length,
        ] = self.streaming_input_buffer[
            :,
            self.hop_length:
        ].clone()

        self.streaming_input_buffer[
            :,
            -self.hop_length:
        ] = audio_hop

        analysis_window = self.analysis_window.to(
            device=audio_hop.device,
            dtype=audio_hop.dtype,
        )

        spectrum_frame = torch.fft.rfft(
            self.streaming_input_buffer * analysis_window,
            n=self.n_fft,
            dim=-1,
        )

        return spectrum_frame.squeeze(0)

    def istft(
        self,
        spectrum: torch.Tensor,
        length: int | None = None,
    ) -> torch.Tensor:
        if spectrum.ndim == 2:
            spectrum = spectrum.unsqueeze(0)

        number_of_frames = spectrum.shape[-1]

        synthesis_window = self.synthesis_window.to(
            device=spectrum.device,
            dtype=spectrum.real.dtype,
        )

        frames = torch.fft.irfft(
            spectrum.transpose(1, 2),
            n=self.n_fft,
            dim=-1,
        )

        frames = frames * synthesis_window

        output_length = (
            self.frame_length
            + (number_of_frames - 1) * self.hop_length
        )

        output = torch.zeros(
            spectrum.shape[0],
            output_length,
            device=frames.device,
            dtype=frames.dtype,
        )

        for frame_index in range(number_of_frames):
            start = frame_index * self.hop_length
            end = start + self.frame_length

            output[:, start:end] += frames[:, frame_index]

        output = output[:, self.left_padding:]

        if length is not None:
            if output.shape[-1] < length:
                output = F.pad(
                    output,
                    (0, length - output.shape[-1]),
                )
            else:
                output = output[:, :length]

        return output.squeeze(0)

    def streaming_istft(
    self,
    spectrum_frame: torch.Tensor,
) -> torch.Tensor:
        if spectrum_frame.ndim == 1:
            spectrum_frame = spectrum_frame.unsqueeze(0)

        synthesis_window = self.synthesis_window.to(
            device=spectrum_frame.device,
            dtype=spectrum_frame.real.dtype,
        )

        frame = torch.fft.irfft(
            spectrum_frame,
            n=self.n_fft,
            dim=-1,
        )

        support_length = 2 * self.hop_length
        support_start = self.frame_length - support_length

        frame = (
            frame[:, support_start:]
            * synthesis_window[support_start:]
        )

        batch_size = frame.shape[0]

        if self.streaming_output_buffer is None:
            self.streaming_output_buffer = torch.zeros(
                batch_size,
                support_length,
                device=frame.device,
                dtype=frame.dtype,
            )

        self.streaming_output_buffer += frame

        output_hop = self.streaming_output_buffer[
            :,
            :self.hop_length,
        ].clone()

        self.streaming_output_buffer[
            :,
            :self.hop_length,
        ] = self.streaming_output_buffer[
            :,
            self.hop_length:
        ]

        self.streaming_output_buffer[
            :,
            self.hop_length:
        ] = 0

        return output_hop.squeeze(0)
    
    

In [ ]:
import math
import torch
from IPython.display import Audio, display


sample_rate = 44100
duration_seconds = 2.0
frequency = 440.0
amplitude = 0.5

number_of_samples = int(sample_rate * duration_seconds)

time = torch.arange(
    number_of_samples,
    dtype=torch.float32,
) / sample_rate

audio = amplitude * torch.sin(
    2.0 * math.pi * frequency * time
)

audio2 = amplitude * torch.sin(
    2.0 * math.pi * frequency*1.83 * time
)

audio +=audio2

audio = torch.cat([torch.zeros_like(audio)[...,:5000], audio])


audio = audio[..., :8192*2]

print(audio.shape)
display(Audio(audio.numpy(), rate=sample_rate))

In [7]:
sample_rate = 44100

analysis_length = 1024
synthesis_length = 128
hop_length = 64

analysis_window, synthesis_window = make_mauler_windows(
    analysis_length=analysis_length,
    synthesis_length=synthesis_length,
    zero_length = 64 ,
    hop_length=hop_length,
)

transform = CausalSTFT(
    analysis_window=analysis_window,
    synthesis_window=synthesis_window,
    hop_length=hop_length,
)

In [ ]:
spectrum, original_length = transform.stft(audio)

reconstructed_audio = transform.istft(
    spectrum,
    length=original_length,
)

display(Audio(audio.numpy(), rate=sample_rate))
display(Audio(reconstructed_audio.numpy(), rate=sample_rate))


# Streaming
transform.reset_streaming_istft()

output_hops = []

for frame_index in range(spectrum.shape[-1]):
    spectrum_frame = spectrum[..., frame_index]
    output_hop = transform.streaming_istft(spectrum_frame)
    output_hops.append(output_hop)

streamed_audio = torch.cat(output_hops, dim=-1)
display(Audio(streamed_audio.numpy(), rate=sample_rate))

# Double Streaming
transform.reset_streaming_stft()
transform.reset_streaming_istft()

output_hops = []
all_spectrum_frames = []

for start in range(0, audio.shape[-1], hop_length):
    audio_hop = audio[start:start + hop_length]

    if audio_hop.shape[-1] < hop_length:
        audio_hop = F.pad(
            audio_hop,
            (0, hop_length - audio_hop.shape[-1]),
        )

    spectrum_frame = transform.streaming_stft(audio_hop)
    output_hop = transform.streaming_istft(spectrum_frame)

    output_hops.append(output_hop)
    all_spectrum_frames.append(spectrum_frame)

double_streamed_audio = torch.cat(output_hops, dim=-1)
all_spectrum_frames = torch.stack(all_spectrum_frames, dim=1)

display(Audio(double_streamed_audio.numpy(), rate=sample_rate))


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Audio, display


original =  audio.cpu().squeeze()
reconstructed = reconstructed_audio.cpu().squeeze()

# Ensure both plots cover exactly the same duration.
plot_length = min(len(original), len(reconstructed))
plot_length = 7000
original = original[4000:plot_length]
reconstructed = reconstructed[4000:plot_length]
streamed_audio = streamed_audio[4000:plot_length]
double_streamed_audio = double_streamed_audio[4000:plot_length]


time = torch.arange(plot_length).numpy() / sample_rate

fig, axes = plt.subplots(
    4,
    1,
    figsize=(14, 10),
    sharex=True,
    sharey=True,
    # gridspec_kw={"hspace": 0.08},
)

axes[0].plot(original, linewidth=0.7)
axes[0].set_title("Original")
axes[0].set_ylabel("Amplitude")
axes[0].grid(alpha=0.25)

axes[1].plot(reconstructed, linewidth=0.7)
axes[1].set_title("iSTFT reconstruction")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Amplitude")
axes[1].grid(alpha=0.25)

axes[2].plot(streamed_audio, linewidth=0.7)
axes[2].set_title("iSTFT reconstruction")
axes[2].set_xlabel("Time (seconds)")
axes[2].set_ylabel("Amplitude")
axes[2].grid(alpha=0.25)

axes[3].plot(double_streamed_audio, linewidth=0.7)
axes[3].set_title("iSTFT reconstruction")
axes[3].set_xlabel("Time (seconds)")
axes[3].set_ylabel("Amplitude")
axes[3].grid(alpha=0.25)

axes[0].set_xlim(800, 1200)
plt.show()

In [ ]:

# CausalMauerSTFT: offline/streaming parity and CPU/GPU timing
import sys
import time
from pathlib import Path

import torch

repo_root = Path.cwd()
if not (repo_root / "after").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from after.autoencoder.audio import CausalMauerSTFT


cfg = dict(nfft=1024, hop_size=64, synthesis_length=128, zero_length=64, normalize=False)
num_samples = 2**16
repeat = 20
warmup = 3


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def bench(fn, device, repeat=repeat, warmup=warmup):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        sync(device)
        start = time.perf_counter()
        for _ in range(repeat):
            fn()
        sync(device)
    return 1000.0 * (time.perf_counter() - start) / repeat


def make_input(device):
    if "audio" in globals():
        x = audio.detach().float().reshape(1, 1, -1)
    else:
        x = torch.randn(1, 1, num_samples)
    x = x[..., :num_samples]
    if x.shape[-1] < num_samples:
        x = torch.nn.functional.pad(x, (0, num_samples - x.shape[-1]))
    return x.to(device)


def stream_encode(transform, x):
    hop = transform.hop_size
    # transform.reset_stream()
    return torch.cat(
        [transform.forward_stream(x[..., i:i + hop])
         for i in range(0, x.shape[-1], hop)],
        dim=-1,
    )


def stream_decode(transform, spec):
    # transform.reset_stream()
    return torch.cat(
        [transform.inverse_stream(spec[..., i:i + 1])
         for i in range(spec.shape[-1])],
        dim=-1,
    )


devices = [torch.device("cpu")]
if torch.cuda.is_available():
    devices.append(torch.device("cuda"))

for device in devices:
    x = make_input(device)
    offline = CausalMauerSTFT(**cfg).to(device).eval()
    stream_stft = CausalMauerSTFT(**cfg).to(device).eval()
    stream_istft = CausalMauerSTFT(**cfg).to(device).eval()

    with torch.inference_mode():
        spec = offline(x)
        stream_spec = stream_encode(stream_stft, x)
        y = offline.inverse(spec)
        stream_y = stream_decode(stream_istft, spec)

    spectrum_err = (spec - stream_spec).abs().max().item()
    raw_audio_err = (y - stream_y).abs().max().item()
    delayed_audio_err = (
        y[..., :-offline.hop_size] - stream_y[..., offline.hop_size:]
    ).abs().max().item()
    with torch.inference_mode():
        encode_ms = bench(lambda: offline(x), device)
        decode_ms = bench(lambda: offline.inverse(spec), device)
        stream_encode_ms = bench(lambda: stream_encode(stream_stft, x), device)
        stream_decode_ms = bench(lambda: stream_decode(stream_istft, spec), device)

    print(f"\nDevice: {device}")
    print(f"spectrum max abs error:       {spectrum_err:.3e}")
    print(f"raw audio max abs error:      {raw_audio_err:.3e}")
    print(f"delayed audio max abs error:  {delayed_audio_err:.3e}")
    print("timing, ms per call")
    print(f"  offline encode:   {encode_ms:8.3f}")
    print(f"  offline decode:   {decode_ms:8.3f}")
    print(f"  streaming encode: {stream_encode_ms:8.3f}")
    print(f"  streaming decode: {stream_decode_ms:8.3f}")


In [ ]:
# StreamableSTFT: CPU/GPU timing
import sys
import time
from pathlib import Path

import torch

repo_root = Path.cwd()
if not (repo_root / "after").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from after.autoencoder.audio import StreamableSTFT


cfg = dict(nfft=1024, hop_size=64)
num_samples = 2**16
repeat = 5
warmup = 3


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def bench(fn, device, repeat=repeat, warmup=warmup):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        sync(device)
        start = time.perf_counter()
        for _ in range(repeat):
            fn()
        sync(device)
    return 1000.0 * (time.perf_counter() - start) / repeat


def make_input(device):
    if "audio" in globals():
        x = audio.detach().float().reshape(1, 1, -1)
    else:
        x = torch.randn(1, 1, num_samples)
    x = x[..., :num_samples]
    if x.shape[-1] < num_samples:
        x = torch.nn.functional.pad(x, (0, num_samples - x.shape[-1]))
    return x.to(device)


def reset_streamable(transform, x):
    if transform.audio_buffer.shape[0] != x.shape[0]:
        transform.audio_buffer = torch.zeros(
            x.shape[0],
            1,
            transform.nfft - transform.hop_size,
            device=x.device,
            dtype=x.dtype,
        )
    transform.audio_buffer.zero_()
    transform.out_buffer.zero_()
    transform.spec_buffer.zero_()


def stream_encode(transform, x):
    hop = transform.hop_size
    reset_streamable(transform, x)
    return torch.cat(
        [transform(x[..., i:i + hop])
         for i in range(0, x.shape[-1], hop)],
        dim=-1,
    )


def stream_decode(transform, spec):
    reset_streamable(transform, spec)
    return torch.cat(
        [transform.inverse_stream(spec[..., i:i + 1])
         for i in range(spec.shape[-1])],
        dim=-1,
    )


devices = [torch.device("cpu")]
if torch.cuda.is_available():
    devices.append(torch.device("cuda"))

for device in devices:
    x = make_input(device)
    offline = StreamableSTFT(**cfg, stream=False).to(device).eval()
    stream_stft = StreamableSTFT(**cfg, stream=True).to(device).eval()
    stream_istft = StreamableSTFT(**cfg, stream=True).to(device).eval()

    with torch.inference_mode():
        spec = offline(x)
        stream_spec = stream_encode(stream_stft, x)
        y = offline.inverse(spec)
        stream_y = stream_decode(stream_istft, stream_spec)

    encode_ms = bench(lambda: offline(x), device)
    decode_ms = bench(lambda: offline.inverse(spec), device)
    stream_encode_ms = bench(lambda: stream_encode(stream_stft, x), device)
    stream_decode_ms = bench(lambda: stream_decode(stream_istft, stream_spec), device)

    print(f"\nDevice: {device}")
    print(f"offline spec shape:   {tuple(spec.shape)}")
    print(f"stream spec shape:    {tuple(stream_spec.shape)}")
    print(f"offline audio shape:  {tuple(y.shape)}")
    print(f"stream audio shape:   {tuple(stream_y.shape)}")
    print("timing, ms per call")
    print(f"  offline encode:   {encode_ms:8.3f}")
    print(f"  offline decode:   {decode_ms:8.3f}")
    print(f"  streaming encode: {stream_encode_ms:8.3f}")
    print(f"  streaming decode: {stream_decode_ms:8.3f}")


In [ ]:
65536/44100
